# State-Specific Upfront Solar Cost Calculation

Calculates state-specific residential upfront solar cost ($/Wdc), for potential use in the analysis.

Two independent methods, **both anchored to the same national price level** (so they sit at the
same overall level and differ only in how they distribute cost across states):

1. **OpenSolar method** — runs the OS reference-case cost model (`reference_case.py` /
   `params.py`, repo root) per state, using LBNL median system size, BEA Regional Price Parity,
   and state sales-tax exemptions (bottom-up), then scales so its install-weighted mean = anchor.
2. **EnergySage method** — takes EnergySage's state-level average $/W (top-down), then scales
   so its install-weighted mean = anchor.

**National anchor** = the 2025-residential-install-weighted **mean** of LBNL's per-state median
$/W (see §1), ≈ **$4.11/W**. Weighting by real 2025 install counts lets California drive the level
at ~its true market share, rather than the coverage-inflated share it holds in LBNL's raw sample.

**LBNL price methodology.** Per-state medians follow LBNL's own price definition — **host-owned
(non-TPO), non-self-installed, PV-only (no battery)**, residential (`RES`/`RES_SF`). Third-party-
owned systems are excluded (their "installed price" is an appraised/transfer value, not an arm's-
length sale, and TPO share is ~41% in CA vs 0% in TX — an uneven bias). This matches LBNL's public
viz tool and pulls the anchor down from an earlier $4.39 (TPO included) to ~$4.11.

**NJ** is redacted in the public LBNL file but hard-coded from the viz tool (~$4.00/W); see §1.

Ends with one CSV: `state, opensolar_method_per_w, energysage_method_per_w, diff`.

In [1]:
import os
import sys

import numpy as np
import pandas as pd

REPO_ROOT = os.path.abspath(os.path.join(os.path.abspath("."), "..", "..", ".."))
DATA_DIR = os.path.join(REPO_ROOT, "data")
sys.path.insert(0, REPO_ROOT)

import params as P
from reference_case import StateInputs, run_all

LBNL_FILE = os.path.join(DATA_DIR, "TTS_LBNL_public_file_29-Sep-2025_all.csv")
MIN_DATE = "2024-01-01"
RES_SEGMENTS = {"RES", "RES_SF"}   # excludes multifamily RES_MF
PRICE_RANGE = (1.0, 10.0)          # $/W plausibility filter for data-entry errors
NATIONAL_MEDIAN_KW = 7.4           # Ramasamy et al. 2022 fallback, states w/o LBNL size coverage

# NJ prices are redacted in the public LBNL file (every NJ row is technology_type="redacted"),
# but LBNL's public viz tool reports a 2024 residential median of ~$4.00/W (n~3,302 host-owned).
# Hard-code that observed value so NJ (a top-5 install market) contributes to the anchor at its
# real level instead of being absent. See "build_baseline_upfront_cost.ipynb" §5.
NJ_VIZ_PRICE_PER_W = 4.00

states_lookup = pd.read_csv(os.path.join(REPO_ROOT, "states.csv"), header=None,
                             names=["state_abbr", "state_full"])

# 2025 residential install counts (weights for both national anchors)
installs_2025 = pd.read_csv(os.path.join(DATA_DIR, "solar_storage_capacity_installations_by_state_sector.csv"))
installs_2025 = installs_2025[
    (installs_2025["year"] == 2025) & (installs_2025["sector"] == "Residential")
][["state", "pv_customers"]].rename(columns={"state": "state_abbr", "pv_customers": "n_installs_2025"})


def weighted_median(values, weights):
    """Value at which cumulative weight first reaches half the total (install-weighted median)."""
    order = np.argsort(values)
    v, w = np.asarray(values)[order], np.asarray(weights)[order]
    return v[np.searchsorted(np.cumsum(w), w.sum() / 2.0)]

## 1) LBNL: state median system size + national price anchor

Same raw file, two uses:
- **Median size by state** (feeds the OpenSolar model's geometry): 2024+, `size > 0`, all
  residential installs (no price/TPO filter — size isn't a price statistic).
- **Per-state median $/W** (LBNL price methodology): 2024+, residential (`RES`/`RES_SF`),
  **PV-only, no battery, host-owned (non-TPO), non-self-installed**, `price_per_w` in `[1, 10]`.
  The `-1` "unknown" ownership sentinel is kept (dropping it would erase TX/MD/DE, which report
  `-1` on every row); only *affirmative* TPO/self-install (`== 1`) is excluded. DE is recovered
  via its `-1` tech-type. **NJ** is redacted in the file → injected from the viz tool ($4.00/W).

**National anchor (option D):** the 2025-residential-install-weighted **mean** of the LBNL state
medians. Weighting by each state's real 2025 install count lets California drive the level at ~its
true market share rather than the reporting-coverage share it occupies in LBNL's raw pooled sample.
A mean (vs a weighted median) avoids the level "snapping" to CA's own median when CA is a ~40%
block. Both methods below are scaled so their 2025-install-weighted mean equals this anchor.

In [2]:
lbnl_cols = ["state", "installation_date", "PV_system_size_DC", "total_installed_price",
             "customer_segment", "technology_type", "third_party_owned", "self_installed",
             "battery_rated_capacity_kWh"]
lbnl = pd.read_csv(LBNL_FILE, usecols=lbnl_cols, encoding="latin-1", low_memory=False)
lbnl["installation_date"] = lbnl["installation_date"].astype(str)
for c in ["PV_system_size_DC", "total_installed_price", "battery_rated_capacity_kWh",
          "third_party_owned", "self_installed"]:
    lbnl[c] = pd.to_numeric(lbnl[c], errors="coerce")
recent = lbnl[lbnl["installation_date"] >= MIN_DATE].copy()

# median system SIZE by state (all residential installs; feeds the OpenSolar model geometry)
lbnl_median_kw = (
    recent[recent["PV_system_size_DC"] > 0]
    .groupby("state")["PV_system_size_DC"].median()
    .rename("median_kw").reset_index().rename(columns={"state": "state_abbr"})
)

# per-state median $/W, using LBNL's PRICE methodology:
#   residential (RES/RES_SF), PV-only + "-1" redacted-techtype (recovers DE), no battery,
#   exclude *affirmative* TPO / self-installed (keep 0 and the -1 "unknown" sentinel so
#   TX/MD/DE aren't erased), price_per_w in [1, 10]. TPO systems report appraised/transfer
#   prices, not arm's-length transactions -- LBNL excludes them from published price stats.
res = recent[
    recent["customer_segment"].isin(RES_SEGMENTS)
    & recent["technology_type"].isin(["pv-only", "-1"])
    & ~(recent["battery_rated_capacity_kWh"] > 0)
    & (recent["third_party_owned"] != 1)
    & (recent["self_installed"] != 1)
    & (recent["PV_system_size_DC"] > 0)
    & (recent["total_installed_price"] > 0)
].copy()
res["price_per_w"] = res["total_installed_price"] / (res["PV_system_size_DC"] * 1000)
res = res[res["price_per_w"].between(*PRICE_RANGE)]
lbnl_price_state_median = (
    res.groupby("state")["price_per_w"].agg(median_price_per_w="median", n="count")
    .reset_index().rename(columns={"state": "state_abbr"})
)

# NJ: redacted in the public file -> inject the LBNL viz-tool observed value (see cell 1).
if "NJ" not in set(lbnl_price_state_median["state_abbr"]):
    lbnl_price_state_median = pd.concat([lbnl_price_state_median, pd.DataFrame(
        [{"state_abbr": "NJ", "median_price_per_w": NJ_VIZ_PRICE_PER_W, "n": 3302}])],
        ignore_index=True)

# NATIONAL ANCHOR (option D): 2025-install-weighted MEAN of the LBNL state medians.
# Weighting by each state's REAL 2025 residential install count lets CA drive the level at
# ~its true market share, NOT the coverage-inflated share it holds in LBNL's raw pooled sample.
anchor = lbnl_price_state_median.merge(installs_2025, on="state_abbr", how="left")
national_anchor = np.average(anchor["median_price_per_w"], weights=anchor["n_installs_2025"])

covered_share = anchor["n_installs_2025"].sum() / installs_2025["n_installs_2025"].sum()
print(f"{len(lbnl_median_kw)}/48 states with LBNL size coverage")
print(f"{len(lbnl_price_state_median)} states with LBNL price coverage "
      f"(incl. NJ hard-code; {covered_share:.0%} of 2025 national residential installs)")
print(f"National anchor (2025-install-weighted mean of LBNL state medians): {national_anchor:.3f}")

23/48 states with LBNL size coverage
18 states with LBNL price coverage (incl. NJ hard-code; 82% of 2025 national residential installs)
National anchor (2025-install-weighted mean of LBNL state medians): 4.112


## 2) OpenSolar method: run the reference-case model, then anchor to LBNL

LBNL median size (else 7.4 kWdc national fallback) x BEA Regional Price Parity x state
sales-tax exemption status. Permit fee / in-person share stay on the national fallback in
`params.py` — SolarTRACE isn't wired in.

The raw model is bottom-up and runs below LBNL actual (a documented property of the methodology
— no margin/competition effects). To put it on the same national footing as the EnergySage
method, we scale the whole vector so its **2025-install-weighted mean matches the national anchor**.
Both methods then sit at the same national level and differ only in cross-state *pattern*.

In [3]:
rpp = pd.read_csv(os.path.join(DATA_DIR, "bea_regional_price_parity.csv")) \
    .rename(columns={"State": "state_full", "Regional Price Parity (2023)": "rpp"})

tax = pd.read_csv(os.path.join(DATA_DIR, "solar_sales_tax_by_state.csv")).rename(columns={"State": "state_full"})
tax["sales_tax_rate"] = tax["State Sales Tax Rate"].str.rstrip("%").astype(float) / 100.0
exempt = tax["Residential Solar Exemption Status"].str.startswith("Exempt") | \
         (tax["Residential Solar Exemption Status"] == "No State Sales Tax")
tax.loc[exempt, "sales_tax_rate"] = 0.0

state_meta = (
    states_lookup.merge(rpp[["state_full", "rpp"]], on="state_full", how="left")
    .merge(tax[["state_full", "sales_tax_rate"]], on="state_full", how="left")
    .merge(lbnl_median_kw, on="state_abbr", how="left")
)
state_meta["median_kw"] = state_meta["median_kw"].fillna(NATIONAL_MEDIAN_KW)

inputs = [
    StateInputs(state=r.state_abbr, median_kw=r.median_kw, rpp=r.rpp, sales_tax_rate=r.sales_tax_rate)
    for r in state_meta.itertuples()
]
opensolar = pd.DataFrame(
    [{"state": state, "opensolar_raw_per_w": total} for state, total, comp, flags in run_all(inputs)]
).merge(installs_2025.rename(columns={"state_abbr": "state"}), on="state", how="left")

# anchor: scale so OpenSolar's 2025-install-weighted mean matches the national anchor
os_raw_wmean = np.average(opensolar["opensolar_raw_per_w"], weights=opensolar["n_installs_2025"])
os_scale = national_anchor / os_raw_wmean
opensolar["opensolar_method_per_w"] = opensolar["opensolar_raw_per_w"] * os_scale

print(f"OpenSolar raw install-weighted mean: {os_raw_wmean:.3f}")
print(f"national anchor:                     {national_anchor:.3f}")
print(f"anchor scale factor:                 {os_scale:.4f}")
opensolar[["state", "opensolar_raw_per_w", "opensolar_method_per_w"]].sort_values("opensolar_method_per_w").head()

OpenSolar raw install-weighted mean: 4.037
national anchor:                     4.112
anchor scale factor:                 1.0188


,state,opensolar_raw_per_w,opensolar_method_per_w
7,OH,3.201777,3.261859
38,UT,3.473167,3.538343
26,MA,3.506363,3.572161
46,VT,3.519406,3.585449
21,IA,3.558353,3.625127


## 3) EnergySage method: scale state $/W to match the national anchor

1. Load EnergySage's state-level average $/W.
2. Compute EnergySage's **2025-install-weighted mean** across states, using each state's 2025
   residential PV customer count as the weight — the same statistic and weights used for the
   national anchor.
3. Scale every state's EnergySage value by `national_anchor / energysage_weighted_mean` — this
   keeps EnergySage's relative cross-state pattern but sets the overall level to match LBNL.

⚠️ EnergySage's low-volume states are noisy (e.g. ND/NE/SD read much higher than any
bottom-up estimate). Install-weighting keeps them from distorting the *anchor*, but those
states still appear inflated in the per-state *output* — treat small-volume states with care.

In [4]:
energysage = pd.read_csv(os.path.join(DATA_DIR, "average_cost_per_watt_energy_sage.csv")) \
    .rename(columns={"State": "state_full"})

es = states_lookup.merge(energysage, on="state_full", how="inner") \
    .merge(installs_2025, on="state_abbr", how="left")

es_weighted_mean = np.average(es["average_cost_per_watt"], weights=es["n_installs_2025"])
scale = national_anchor / es_weighted_mean
es["energysage_method_per_w"] = es["average_cost_per_watt"] * scale

print(f"EnergySage 2025-install-weighted mean: {es_weighted_mean:.3f}")
print(f"national anchor:                       {national_anchor:.3f}")
print(f"scale factor:                          {scale:.4f}")
es[["state_abbr", "average_cost_per_watt", "n_installs_2025", "energysage_method_per_w"]].sort_values("energysage_method_per_w").head()

EnergySage 2025-install-weighted mean: 2.564
national anchor:                       4.112
scale factor:                          1.6036


,state_abbr,average_cost_per_watt,n_installs_2025,energysage_method_per_w
0,TX,2.17,221661.0,3.479877
2,FL,2.20,263388.0,3.527985
23,AZ,2.23,328287.0,3.576094
9,NC,2.26,58723.0,3.624203
29,AR,2.43,16021.0,3.896820


## 4) Combine and export

In [5]:
comparison = opensolar[["state", "opensolar_method_per_w"]].merge(
    es[["state_abbr", "energysage_method_per_w"]].rename(columns={"state_abbr": "state"}),
    on="state", how="outer"
).sort_values("state").reset_index(drop=True)

comparison["diff"] = comparison["energysage_method_per_w"] - comparison["opensolar_method_per_w"]

# both anchored on 2025-install-weighted mean, so that stat should be ~national_anchor for each;
# the unweighted median differs by cross-state pattern only
w = opensolar.set_index("state")["n_installs_2025"]
for col in ["opensolar_method_per_w", "energysage_method_per_w"]:
    s = comparison.set_index("state")[col]
    ww = w.reindex(s.index)
    print(f"{col}: unweighted median {s.median():.3f}, "
          f"install-weighted mean {np.average(s, weights=ww):.3f}")

out_path = os.path.join(DATA_DIR, "state_upfront_cost_opensolar_vs_energysage.csv")
comparison.to_csv(out_path, index=False)
print("\nSaved to", out_path)
comparison

opensolar_method_per_w: unweighted median 3.767, install-weighted mean 4.112
energysage_method_per_w: unweighted median 4.330, install-weighted mean 4.112

Saved to /Users/wael/Documents/repos/dgen/data/state_upfront_cost_opensolar_vs_energysage.csv


,state,opensolar_method_per_w,energysage_method_per_w,diff
0,AL,3.719799,5.484414,1.764615
1,AR,3.688016,3.896820,0.208804
2,AZ,3.774420,3.576094,-0.198326
3,CA,4.504942,4.041147,-0.463795
4,CO,4.062689,4.361873,0.299184
5,CT,3.980275,4.329800,0.349525
6,DE,3.714694,4.137365,0.422670
7,FL,3.800375,3.527985,-0.272389
8,GA,3.863182,3.944929,0.081747
9,IA,3.625127,4.987288,1.362161
